# 02 — Two-stage anchor-based detector: Faster R-CNN (torchvision)

Faster R-CNN ResNet-50-FPN, COCO-pretrained, new 9-class box predictor (8 classes + background).
**Stage 1 (RPN)** scores ~180k anchors per frame as object / background and keeps the best
proposals after NMS. **Stage 2 (RoI head)** pools features for each proposal, classifies it into the
8 classes and refines the box.

| Run | Change | Why |
|---|---|---|
| `frcnn_base` | torchvision defaults: 2000 / 1000 proposals (train / test), 512 sampled RoIs per frame with 25% positives, Smooth-L1 box loss | baseline |
| `frcnn_dense` | 3000 / 2000 proposals, 50% positive RoIs | dense scenes (≈46 objects per frame, up to 180): more proposals and more positive samples per frame |
| `frcnn_giou` | better of the two + **GIoU** box loss (weight 10) | loss modification: GIoU is scale-invariant, so small boxes are not under-weighted |

After training we also check, without retraining, how the number of test-time proposals changes
mAP and speed, and how many objects the RPN alone already covers (proposal recall).

Approximate time on a T4 with `FAST=True`: ~0.5 h per run, ~1.7 h in total (~4 h without FAST).

## 0. Setup
Works on **Kaggle** (add the competition as input, GPU, Internet on), **Google Colab** (GPU T4,
Kaggle token as Colab secret `KAGGLE_API_TOKEN`, or `kaggle.json` in the root of Google Drive) and **a laptop** (Kaggle token in `<repo>/.kaggle_token`, git-ignored;
data and results go to `~/vdet_data` and `~/vdet_work`; without a GPU use it only with `SMOKE = True`).
On Colab all runs are saved to `MyDrive/vdet`, so after a disconnect just run the notebook again:
finished runs are skipped and interrupted runs continue from the last epoch.

In [ ]:
# ---- Settings ----
SMOKE = False   # True: tiny run on a few images, only to check that the notebook works end to end
FAST = True     # True: every 20th train frame and fewer epochs (~2x faster); False: full protocol
REPO = "https://github.com/LidaSokha03/AI_HW2.git"   # shared code is in src/ of this repository
PRETRAINED = True   # COCO weights from torchvision

In [ ]:
import os
import sys
import subprocess

if "google.colab" in sys.modules:  # Colab: keep runs and results on Google Drive
    from google.colab import drive
    drive.mount("/content/drive")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'pycocotools', 'kaggle>=2.0'], check=True)

# Shared code (src/vdet.py, src/tvdet.py): local copy if present, otherwise cloned from GitHub
CANDIDATES = ["src", "../src", ".", "notebooks"]
SRC = os.environ.get("VDET_SRC") or next((p for p in CANDIDATES if os.path.exists(f"{p}/vdet.py")), None)
if SRC is None:
    if os.path.exists("/tmp/AI_HW2"):
        subprocess.run(["git", "-C", "/tmp/AI_HW2", "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "/tmp/AI_HW2"], check=True)
    SRC = next(f"/tmp/AI_HW2/{p}" for p in ["src", "notebooks"] if os.path.exists(f"/tmp/AI_HW2/{p}/vdet.py"))
sys.path.insert(0, os.path.abspath(SRC))

import vdet

# Same frame subset for every model (see the protocol below). First run: ~5 min to prepare.
TRAIN_STEP = 20 if FAST else 10
S = vdet.setup(train_step=TRAIN_STEP, val_step=5, max_side=1600, smoke=SMOKE)
print({k: S[k] for k in ["platform", "work", "prep"]})

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
import tvdet

NB = "02_faster_rcnn"
GT = f"{S['prep']}/val_gt.json"
VAL = vdet.val_items(S)[:20] if SMOKE else vdet.val_items(S)
TEST = vdet.test_items(S)[:30] if SMOKE else vdet.test_items(S)
COLS = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR500", "ms_per_img", "FPS"]
print("device:", tvdet.DEVICE)

### Common protocol (identical in all model notebooks)
| Item | Value |
|---|---|
| Split | given train / val split, which is already **by video** (56 / 7 videos, no leakage) |
| Frames | every 20th train frame (≈1,200, `FAST=True`) or every 10th (≈2,400) and every 5th val frame (≈570): neighbouring frames are near-duplicates |
| Resolution | frames stored with longer side ≤ 1600 px; model input longer side **1280 px** unless the experiment changes it |
| Targets | `iscrowd=1` boxes (ignored regions, "others") are removed from training targets |
| Metric | COCO mAP@[.5:.95], AP50, AP_small… with `pycocotools` on val at **original resolution**, ignored regions as `iscrowd`, ≤ 500 detections per frame (same as the leaderboard) |
| Speed | ms per frame on the same GPU, batch 1, including image reading and NMS |
| Budget | ≈ 0.5 GPU-hour (T4) per run with `FAST=True`, ≈ 1 hour without |
| Seed | 42 |

### Training settings
SGD, momentum 0.9, weight decay 1e-4, batch 4, lr 0.005 (torchvision reference 0.02 for batch 16,
scaled linearly), 300 warm-up iterations, 4 epochs (6 without FAST) with lr × 0.1 for the last epoch, mixed precision,
gradient clipping 10, horizontal flip. Validation mAP after every epoch; the best epoch is kept.

In [ ]:
EPOCHS = 1 if SMOKE else (4 if FAST else 6)
TRAIN_KW = dict(epochs=EPOCHS, batch_size=4, lr=0.005,
                max_iters_per_epoch=3 if SMOKE else None, eval_limit=20 if SMOKE else None)
CONFIGS = {
    "frcnn_base": dict(),
    "frcnn_dense": dict(rpn_pre_nms_top_n_train=3000, rpn_post_nms_top_n_train=3000,
                        rpn_pre_nms_top_n_test=2000, rpn_post_nms_top_n_test=2000,
                        box_positive_fraction=0.5),
}
results, val_preds, histories = {}, {}, {}


def run_frcnn(name, cfg, box_loss="smooth_l1"):
    """Train (or resume / reuse) one run, evaluate the best epoch on val and log the result."""
    tvdet.set_frcnn_box_loss(box_loss)
    model = tvdet.build_faster_rcnn(pretrained=PRETRAINED, **cfg)
    histories[name] = tvdet.train(model, S, name, **TRAIN_KW)
    tvdet.set_frcnn_box_loss("smooth_l1")
    model = tvdet.load_best(model, S, name)
    preds = tvdet.predict(model, VAL)
    metrics = vdet.coco_eval(GT, preds, per_class=True)
    metrics.update(tvdet.speed(model, VAL, n=5 if SMOKE else 100))
    vdet.log_result(S, NB, name, metrics, epochs=EPOCHS, box_loss=box_loss, config=str(cfg))
    results[name], val_preds[name] = metrics, preds
    return model, pd.DataFrame([metrics], index=[name])[COLS]

## 1. Baseline

In [ ]:
model, row = run_frcnn("frcnn_base", CONFIGS["frcnn_base"])
row

## 2. Experiment 1: more proposals and more positive RoIs (dense scenes)

In [ ]:
model, row = run_frcnn("frcnn_dense", CONFIGS["frcnn_dense"])
row

## 3. Experiment 2: GIoU box loss (on top of the better configuration)

In [ ]:
base_cfg = max(["frcnn_base", "frcnn_dense"], key=lambda n: results[n]["mAP"])
print("GIoU run uses the configuration of:", base_cfg)
CONFIGS["frcnn_giou"] = CONFIGS[base_cfg]
model, row = run_frcnn("frcnn_giou", CONFIGS["frcnn_giou"], box_loss="giou")
row

## 4. Comparison

In [ ]:
table = vdet.results_table(results)
display(table[COLS])
vdet.plot_size_bars(table, "Faster R-CNN: AP by object size", save=f"{S['work']}/figs/frcnn_ap_by_size.png")
print("AP per class:")
display(table[[f"AP_{c}" for c in vdet.CLASS_NAMES]].astype(float).round(3))

fig, ax = plt.subplots(figsize=(7, 4))
for name, h in histories.items():
    ax.plot(h["epoch"], h["mAP"], marker="o", label=name)
ax.set_xlabel("epoch"); ax.set_ylabel("val mAP"); ax.set_title("Faster R-CNN: val mAP per epoch"); ax.legend()
plt.tight_layout(); plt.savefig(f"{S['work']}/figs/frcnn_curves.png", dpi=110); plt.show()

## 5. What each stage contributes
**RPN recall**: share of ground-truth objects that have at least one proposal with IoU ≥ 0.5,
for the top-N proposals. If an object is not covered here, the second stage cannot detect it.
**Final detections**: the second stage turns proposals into class-specific, refined boxes and
removes most of them (see precision / error types below).

In [ ]:
best = table["mAP"].astype(float).idxmax()
print("best run:", best)
tvdet.set_frcnn_box_loss("smooth_l1")
best_model = tvdet.load_best(tvdet.build_faster_rcnn(pretrained=False, **CONFIGS[best]), S, best)
print("RPN proposal recall (IoU >= 0.5) by object size:")
display(tvdet.rpn_recall(best_model, S, VAL, topn=(100, 300, 1000, 2000)))

Number of test-time proposals (no retraining): effect on accuracy and speed.

In [ ]:
rows = {}
for n in [300, 1000, 2000]:
    best_model.rpn._pre_nms_top_n["testing"] = max(n, 1000)
    best_model.rpn._post_nms_top_n["testing"] = n
    m = vdet.coco_eval(GT, tvdet.predict(best_model, VAL))
    m.update(tvdet.speed(best_model, VAL, n=5 if SMOKE else 50))
    rows[f"{n} proposals"] = m
best_model.rpn._pre_nms_top_n["testing"] = CONFIGS[best].get("rpn_pre_nms_top_n_test", 1000)
best_model.rpn._post_nms_top_n["testing"] = CONFIGS[best].get("rpn_post_nms_top_n_test", 1000)
display(vdet.results_table(rows)[["mAP", "AP50", "AP_small", "AR500", "ms_per_img"]])

## 6. Failure cases
Error types at score ≥ 0.3 (TIDE-like): duplicate = second box on an already found object,
classification = right place / wrong class, localization = right class / IoU 0.1–0.5,
background = no object nearby. Below: the val frames with the most errors (green = correct,
red = false positive, yellow = missed object, grey = ignored region).

In [ ]:
miss = vdet.error_breakdown(S, val_preds[best])
ids = vdet.worst_frames(S, val_preds[best], k=4)
vdet.plot_errors(S, val_preds[best], ids, save=f"{S['work']}/figs/frcnn_failures.png")

## 7. Submission with the best run

In [ ]:
vdet.write_submission(S, tvdet.predict(best_model, TEST), best)